# 任务 3：Dorado → HTTP GPU worker → HDFS

新建独立 Notebook 使用此入口，不覆盖已有推理 Notebook。GPU 实例只用平台模型目录和本地盘。控制端须具备 CPU PyTorch、bytedmerlin、HDFS CLI 和到 GPU PORT0 的网络权限。

运行前：平台绑定完整离线包、设置有限 Spot-UCE 到期时间；在控制端环境配置 LOWBAND_PACKAGE_DIR（已解包目录）、LOWBAND_RUN_ID（恢复时保持不变）、LOWBAND_HTTP_TOKEN（与两个 worker 相同的密钥）。不要把密钥直接写进 Notebook。

这份 Notebook 尚未在 Dorado 执行。按顺序运行各 cell，预检失败将停止训练入口。


In [ ]:
import json, os, sys
from pathlib import Path
package = Path(os.environ['LOWBAND_PACKAGE_DIR']).resolve()
assert (package / 'package-manifest.json').is_file(), 'Expected the unpacked offline package'
sys.path.insert(0, str(package))
run_id = os.environ['LOWBAND_RUN_ID']
token = os.environ['LOWBAND_HTTP_TOKEN']
assert len(token) >= 24
from lowband_training.dorado_discovery import BernardEndpoints
from lowband_training.http_probe import probe
services = {
    'psm': '<configured-psm>',
    'services': {'hl':'<configured-service>', 'lf':'<configured-service>'},
    'region': 'cn', 'cluster': ['default'], 'env_type': ['prod'], 'refresh_seconds': 30,
}
provider = BernardEndpoints(**services)
endpoints = provider()
print({'discovered_endpoint_counts': provider.counts, 'selected_slots': sorted(endpoints)})
assert set(endpoints) == {'hl', 'lf'}, 'Wait for both task-3 services to receive resources'


In [ ]:
probe_reports = {}
for slot, endpoint in endpoints.items():
    result = probe(endpoint, token,
        '<controller-storage-uri><operator>/lowband_training/http-validation')
    assert result['http_verified'] and result['hdfs_verified'] and result['worker'] == slot
    probe_reports[slot] = result
print(json.dumps(probe_reports, indent=2))


## 训练控制循环

只允许一个活跃控制进程；不要把这个 cell 放进 mapPartitions 或并发执行两份。HTTP worker 默认每 120 秒需要续租，训练在后台子进程进行。所有结果取回并写入 HDFS 后才提交轮次。控制端重启时用相同 run_id/config 恢复。


In [ ]:
import torch
from lowband_training.hdfs_store import HdfsStore
from lowband_training.http_controller import Controller
assert set(probe_reports) == {'hl', 'lf'}
torch.set_num_threads(2)
config = json.loads((package / 'config.json').read_text())
store = HdfsStore(
    '<controller-storage-uri><operator>/lowband_training/runs/' + run_id,
    '/tmp/lowband-controller-' + run_id,
)
controller = Controller(store, config, provider, token, package / 'assets', run_id)
completed = controller.run()
print(json.dumps(completed, indent=2))


## 结束

completed.json 表示训练状态已提交，不能代表 Bernard 副本已释放。回读两张限时预约及 GPU 实例状态，按每站最多三小时的预算结束资源。正式质量评估和吞吐比较需要独立记录，不能用这次短训练宣称模型收敛或跨机加速。
